---
## Section 1 — Data Acquisition with `owilix`

### What is the Open Web Index (OWI)?

The [**Open Web Index (OWI)**](https://openwebsearcheu-public.pages.it4i.eu/owi-cli/index.html) is a European open-source web crawl — think of it as an open, transparent alternative to access the web. The `owilix` command-line tool lets you download and search that index from your own machine.

### Two paths to get data

```

PATH A — Download first using owilix > Save CIFF + Parquet files > Build local MOSAIC index

┌─────────────────────────────────────────────────────────────────────┐
│                        Your Question / Query                        │
└──────────────────────┬──────────────────────────┬───────────────────┘
                       │                          │
                       │                    PATH B — Live remote search (still very slow)
                       │                          │
                       │                    owilix remote search "..."
                       │                          │   
                       │                    Results returned immediately
                       │                          │                        
                       │                          │
           Path A -Fast offline retrieval   No local index needed
                       └──────────────┬───────────┘
                                      │
                                 LLM Answer
                                 (Section 5)
```

| | **Path A — Local** | **Path B — Remote** |
|---|---|---|
| Speed after setup | Very fast | Depends on network |
| Works offline | ✅ | ❌ |
| Setup effort | Higher (index build) | Minimal |
| Custom data | ✅ | ❌ |



### Installing `owilix`

Run the following **in your terminal** (with the `opensearch_hackathon` venv active) before continuing in the notebook.

```bash
curl -fsSL https://openwebindex.net/owilix/install.sh | sh
```

When prompted, type the number for your package manager (`1` for conda, `2` for mamba) and press Enter.

After installation, verify it works:
```bash
owilix --version
```

Then come back here and run the next cell.

In [11]:
#  TEST — verify owilix is installed and reachable
import subprocess, pathlib

def test_owilix_installed() -> bool:
    """Return True if owilix is available on PATH and reports a version."""
    result = subprocess.run(["owilix", "--version"], capture_output=True, text=True)
    if result.returncode == 0:
        print("owilix version:", result.stdout.strip())
        print("✅ owilix is installed and on PATH")
        return True
    print("❌ owilix not found. Re-run the install cell above, then restart this notebook.")
    return False

owilix_ok = test_owilix_installed()

owilix version: owilix 5.5.1
✅ owilix is installed and on PATH


In [12]:
#  1.3  List available remote datasets
# Shows what OWI snapshots are available for download or live search.
# to have access, you will be redirected to the login page: choose the B2ACCESS on the right side, then scroll down and choose *DLR - Deutsches Zentrum..."

if owilix_ok:
    result = subprocess.run(["owilix", "remote", "ls", "all"], capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    print("⚠️  Skipped — owilix not installed")

# YOU CAN ALSO Fetch for a specific dataset 
# or in command line owilix remote pull all/id=42b3286c-5ad2-11f1-ae03-0e7110c447b2
if owilix_ok:
    result = subprocess.run(["owilix", "remote", "ls", "all/id=42b3286c-5ad2-11f1-ae03-0e7110c447b2"], capture_output=True, text=True)
    print(result.stdout or result.stderr)

📦      d1b020e8-5f3b-11f1-b1e0-0e7110c447b2    OWI-Open Web 
Index-curlie_full.owi@it4i-2026-06-01:2026-06-01   2026-06-01      curlie_full  
lexis   IT4ILexisV2             #=1,445,103     8.18GiB public
📦      92a01354-5f3b-11f1-bcbf-0e7110c447b2    OWI-Open Web 
Index-licenses.owi@it4i-2026-06-01:2026-06-01      2026-06-01      licenses     
lexis   IT4ILexisV2             #=5,790 0.04GiB public
📦      67f076a8-5f3b-11f1-8a15-0e7110c447b2    OWI-Open Web 
Index-main.owi@it4i-2026-06-01:2026-06-01  2026-06-01      main    lexis   
IT4ILexisV2             #=17,181,836    96.17GiB        public
📦      67ea13d0-5f3b-11f1-ae03-0e7110c447b2    OWI-Open Web 
Index-legal.owi@it4i-2026-06-01:2026-06-01 2026-06-01      legal   lexis   
IT4ILexisV2             #=195,609       0.84GiB public
📦      fb121768-5f39-11f1-b1e0-0e7110c447b2    OWI-Open Web 
Index-main.owi@it4i-2026-05-31:2026-05-31  2026-05-31      main    lexis   
IT4ILexisV2             #=17,892,231    99.83GiB        public
📦     

###  1.5  PATH A — Download data locally 

The downloaded files (CIFF + Parquet) will be used in Section 3 to build
a local MOSAIC search index.


For example run 

'owilix remote pull all/id=42b3286c-5ad2-11f1-ae03-0e7110c447b2'

 (see https://openwebindex.eu/owler/our_datasets)

### Let us take a look at the data

In [14]:
# What is in the data
# ls ~/.owi/public/curlie_full/42b3286c-5ad2-11f1-ae03-0e7110c447b2/year=2026/month=6/day=1/language=eng/    
# 
# 
import pandas as pd 
from glob import glob
import os

# USUALLY, by default the data is downloaded to your home directory/.owi
home = os.path.expanduser("~")
parquets = glob(f"{home}/.owi/public/curlie_full/42b3286c-5ad2-11f1-ae03-0e7110c447b2/*/*/*/language=eng/*.parquet")

# Let us read a sample parquet file to inspect what is there
sample_parquet = pd.read_parquet(parquets[0])
sample_parquet[["mime_type", "curlielabels_en",]].head(2)

# uncomment to check curlie label: print(sample_parquet["curlielabels_en"].iloc[0], type(sample_parquet["curlielabels_en"].iloc[0]))
#. uncomment to check all columns: sample_parquet.columns.to_list()

sample_parquet[['id',
 'record_id',
 'title',
 'description',
 'keywords',
 'author',
 'main_content',
 'content_length',
 'json-ld',
 'microdata',
 'url',
  'url_domain',
   'language',
   'curlielabels_en',
   ]].head(2)

,id,record_id,title,description,keywords,author,main_content,content_length,json-ld,microdata,url,url_domain,language,curlielabels_en
0,86696be290ace72e7e033d134028397c0cf0cb94c6c0cb...,0731e889-d96d-4fa9-b94e-a3806f10236b,"Color: Pink, Height or Size: 60%2522-~-63%2522",Discover the epTour Lite Driver from U.S. Kids...,"elite junior golf, kids golf, kids golf equipm...",NaN,"<a href=""#"">Back</a>\n\n<h1>epTOUR</h1>\n\n<h2...",396,NaN,"[{'type': 'https://schema.org/WebPage', 'prope...",https://www.uskidsgolf.com/eptour/custitem_cus...,uskidsgolf,eng,[Business/Consumer_Goods_and_Services/Sporting...
1,86699419d35ff05e900e48e599ad1437dd3f2a762affd3...,e9446501-df8e-4680-bbe3-889aa2c4cfd0,Lithuania – Daily Sundial,The student media organization of California S...,NaN,NaN,"<a href=""https://sundial.csun.edu/"">\nThe stud...",868,"[{'@context': 'http://schema.org', '@type': 'E...",NaN,https://sundial.csun.edu/tag/lithuania/,csun,eng,[News/Colleges_and_Universities/Newspapers/Uni...


---
## Data Processing

Raw web crawl records are messy. Pages contain navigation menus, cookie banners, ads, and boilerplate HTML that is useless for search and wastes tokens when sent to the LLM.

For thsi hackathon, we use basic filtering, relying on OWI columns

In [ ]:
## Here we only need the Parquet file to inspect and clean the page content.

def filter_data(df, min_content_length=500, label_keyword="Science"):
    """
    Filter the raw OWI dataframe down to usable, on-topic records.

    Keeps rows where:
      - mime_type is "text/html"
      - content_length >= min_content_length
      - curlielabels_en contains `label_keyword` (case-insensitive),
        safely handling None / NaN / empty arrays / empty strings
    """
    def has_label(labels, keyword):
        keyword = keyword.lower()

        # None
        if labels is None:
            return False

        # NaN (float) — pd.isna also handles None, but guard scalars only
        if isinstance(labels, float) and pd.isna(labels):
            return False

        # numpy array / list-like
        if hasattr(labels, "__iter__") and not isinstance(labels, str):
            if len(labels) == 0:
                return False
            return any(
                keyword in str(l).lower()
                for l in labels
                if l is not None and str(l).strip() != "" and not (isinstance(l, float) and pd.isna(l))
            )

        # plain string
        labels_str = str(labels).strip()
        if labels_str == "" or labels_str.lower() == "nan":
            return False
        return keyword in labels_str.lower()

    mask = (
        (df["mime_type"] == "text/html") &
        (df["content_length"] >= min_content_length) &
        (df["curlielabels_en"].apply(lambda x: has_label(x, label_keyword)))
    )

    cleaned_df = df[mask].reset_index(drop=True)

    print(f"Filtered {len(df)} → {len(cleaned_df)} records ({round((len(cleaned_df)*100)/len(df), 2)}% kept) "
          f"(mime_type=text/html, content_length>={min_content_length}, "
          f"label contains '{label_keyword}')")
    return cleaned_df



def filter_all_data(output_dir="data/", filename="filtered_metadata.parquet", force_create: bool = False):

    print(f"There are {len(parquets)} files.")
    output_path = os.path.join(output_dir, filename)

    if os.path.exists(output_path) and not force_create:
        print(f"Found existing file, loading: {output_path}")
        combined_df = pd.read_parquet(output_path)
        print(f"   Loaded {len(combined_df)} records")
        return combined_df

    filtered_dfs = []

    for f in parquets:
        df = pd.read_parquet(f)  
        cleaned_df = filter_data(df, min_content_length=500, label_keyword="Science")      
        if len(cleaned_df) > 0:
            filtered_dfs.append(cleaned_df)
    if len(filtered_dfs)> 0 :
        combined_df = pd.concat(filtered_dfs, ignore_index=True)

        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, filename)
        combined_df.to_parquet(output_path, index=False)
        print("Total records: ", len(combined_df))
    return combined_df


filtered_metadata_df = filter_all_data(force_create=True, )

There are 8 files.
Filtered 13373 → 870 records (6.51% kept) (mime_type=text/html, content_length>=500, label contains 'Science')
Filtered 11960 → 765 records (6.4% kept) (mime_type=text/html, content_length>=500, label contains 'Science')
Filtered 2914 → 167 records (5.73% kept) (mime_type=text/html, content_length>=500, label contains 'Science')
Filtered 12831 → 811 records (6.32% kept) (mime_type=text/html, content_length>=500, label contains 'Science')
Filtered 12016 → 756 records (6.29% kept) (mime_type=text/html, content_length>=500, label contains 'Science')
Filtered 13111 → 809 records (6.17% kept) (mime_type=text/html, content_length>=500, label contains 'Science')
Filtered 7314 → 447 records (6.11% kept) (mime_type=text/html, content_length>=500, label contains 'Science')
Filtered 12531 → 760 records (6.06% kept) (mime_type=text/html, content_length>=500, label contains 'Science')
Total records:  5385
